<a href="https://www.kaggle.com/code/martinsertin/blueberry-yield-optuna-xgb?scriptVersionId=287610378" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')

test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])


In [3]:
def create_features(df, kmeans=None, scaler=None, fit=False):
    data = df.copy()

    data['total_bees'] = data[['honeybee','bumbles','andrena','osmia']].sum(axis=1)
    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['bee_inter'] = data['osmia'] * data['honeybee']

    data['temp_range'] = data['MaxOfUpperTRange'] - data['MinOfLowerTRange']
    data['avg_temp'] = (data['AverageOfUpperTRange'] + data['AverageOfLowerTRange']) / 2
    data['temp_rain'] = data['avg_temp'] * data['RainingDays']

    for col in ['clonesize','total_bees','fruitmass','seeds']:
        data[f'log_{col}'] = np.log1p(data[col])

    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)

    cluster_cols = ['clonesize','total_bees','avg_temp','RainingDays']
    if fit:
        scaler = StandardScaler()
        Xs = scaler.fit_transform(data[cluster_cols])
        kmeans = KMeans(n_clusters=6, n_init=20, random_state=42)
        data['cluster'] = kmeans.fit_predict(Xs)
    else:
        Xs = scaler.transform(data[cluster_cols])
        data['cluster'] = kmeans.predict(Xs)

    return data, kmeans, scaler

In [4]:
train_fe, kmeans, scaler = create_features(train, fit=True)
test_fe, _, _ = create_features(test, kmeans=kmeans, scaler=scaler)

X = train_fe.drop(columns=['yield'])
y = np.log1p(train_fe['yield'])

y_true = train_fe['yield'].values
y_min, y_max = y_true.min(), y_true.max()

In [5]:
def xgb_model():
    return XGBRegressor(
        n_estimators=7000,
        learning_rate=0.025,
        max_depth=6,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.15,
        reg_alpha=0.3,
        reg_lambda=2.5,
        objective='reg:absoluteerror',
        tree_method='gpu_hist',
        predictor='gpu_predictor',
        random_state=42
    )

def cat_model():
    return CatBoostRegressor(
        iterations=7000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=6,
        loss_function='MAE',
        eval_metric='MAE',
        task_type='GPU',
        random_seed=42,
        verbose=False
    )

def meta_model():
    return XGBRegressor(
        n_estimators=4000,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='reg:absoluteerror',
        tree_method='gpu_hist',
        predictor='gpu_predictor',
        random_state=42
    )

In [6]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
test_xgb = np.zeros(len(test_fe))
test_cat = np.zeros(len(test_fe))

for fold, (tr, val) in enumerate(kf.split(X), 1):
    print(f"Level-1 Fold {fold}/10")

    # XGB
    xgb = xgb_model()
    xgb.fit(
        X.iloc[tr], y.iloc[tr],
        eval_set=[(X.iloc[val], y.iloc[val])],
        early_stopping_rounds=300,
        verbose=False
    )
    oof_xgb[val] = np.expm1(xgb.predict(X.iloc[val]))
    test_xgb += np.expm1(xgb.predict(test_fe)) / kf.n_splits

    # CAT
    cat = cat_model()
    cat.fit(
        X.iloc[tr], y.iloc[tr],
        eval_set=(X.iloc[val], y.iloc[val]),
        use_best_model=True
    )
    oof_cat[val] = np.expm1(cat.predict(X.iloc[val]))
    test_cat += np.expm1(cat.predict(test_fe)) / kf.n_splits

Level-1 Fold 1/10


Default metric period is 5 because MAE is/are not implemented for GPU


Level-1 Fold 2/10


Default metric period is 5 because MAE is/are not implemented for GPU


Level-1 Fold 3/10


...

Default metric period is 5 because MAE is/are not implemented for GPU


Level-1 Fold 9/10


Default metric period is 5 because MAE is/are not implemented for GPU


Level-1 Fold 10/10


Default metric period is 5 because MAE is/are not implemented for GPU


In [7]:
meta_X = np.column_stack([oof_xgb, oof_cat])
meta_test = np.column_stack([test_xgb, test_cat])

meta = meta_model()
meta.fit(meta_X, y_true)

oof_stack = meta.predict(meta_X)
test_stack = meta.predict(meta_test)

mae = mean_absolute_error(y_true, oof_stack)
print(f"\n🏆 FINAL STACK MAE: {mae:.4f}")


🏆 FINAL STACK MAE: 239.8419


In [8]:
test_stack = np.clip(test_stack, y_min, y_max)

submission = pd.DataFrame({
    'id': test_ids,
    'yield': test_stack
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,yield
0,15000,7510.119141
1,15001,5895.058105
2,15002,6526.368164
3,15003,4546.526367
4,15004,5871.062012
